# 02 - Training, baselines and ablation (E01-10, #28)
Five systems under one protocol (rules, SQL-only, data-only, hybrid, hybrid + semantic constraints), leave-one-group-out ablation,
synthetic->real transfer on the LLM-labelled pilot. **Every number states what it measures.** No definitive model is trained here:
contract readings are confirmed (FEATURE_INTERPRETATIONS.md pinned in every report) but human validation is pending. Requires outputs of notebook 01 in `SQLTOVIZ_OUT`.

In [ ]:
import os, sys, json, subprocess, pathlib, time
MODE = os.environ.get("SQLTOVIZ_MODE", "smoke")            # "smoke" (minutes, subset) | "full"
SEED = 20261127
COMMIT = os.environ.get("SQLTOVIZ_COMMIT", "development")   # pin a commit SHA for publishable runs
REPO = os.environ.get("SQLTOVIZ_REPO")
if not REPO:                                                # Kaggle: clone the repository at the pinned commit
    REPO = "/kaggle/working/sqltoviz"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "https://github.com/jmmana/sqltoviz", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", COMMIT], check=True)
sys.path.insert(0, f"{REPO}/research/kaggle")
OUT = os.environ.get("SQLTOVIZ_OUT", "/kaggle/working/out"); os.makedirs(OUT, exist_ok=True)
DATA = os.environ.get("SQLTOVIZ_DATA", "/kaggle/working/data")
GIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("mode", MODE, "| repo commit", GIT, "| out", OUT)
from sqltoviz_data import features as F
F.load_contract()                                           # raises if the contract bytes differ from the pinned SHA256
print(F.CONTRACT_VERSION, F.CONTRACT_SHA256)

In [ ]:
import numpy as np, pathlib
from sqltoviz_data import pipeline as P, splits as S, experiment as E, rules as R
for d in ("reports",): os.makedirs(f"{OUT}/{d}", exist_ok=True)
syn = [json.loads(l) for l in open(f"{OUT}/apache-2.0/synthetic.jsonl")]
fs = P.load_features(f"{OUT}/apache-2.0/features_synthetic.npz")          # raises if built with another contract
byid = {it["item_id"]: it for it in syn}
syn = [byid[i] for i in fs["item_ids"]]
X, C = fs["table"], fs["columns"]
print(len(syn), "synthetic items |", F.CONTRACT_VERSION, F.CONTRACT_SHA256[:12])

## E1 - synthetic -> held-out synthetic schemas (agreement with the generator convention; NOT real accuracy)

In [ ]:
SEEDS = [0] if MODE == "smoke" else [0, 1, 2]
idx = {it["item_id"]: k for k, it in enumerate(syn)}
runs = []
for sd in SEEDS:
    assign = S.grouped_stratified_split(syn, group=lambda x: x["db_id"], seed=sd)
    S.audit_splits(syn, assign, group=lambda x: x["db_id"])
    tr = [i for i in syn if assign[i["item_id"]] == "train"]; te = [i for i in syn if assign[i["item_id"]] == "test"]
    tri, tei = [idx[i["item_id"]] for i in tr], [idx[i["item_id"]] for i in te]
    r = E.evaluate_all(tr, te, seed=sd, Xtr=X[tri], Xte=X[tei], cols_te=[C[k] for k in tei]); r["split_seed"] = sd; runs.append(r)
for s in E.SYSTEMS:
    print(f"{s:16s} macroF1={np.mean([r['systems'][s]['macro_f1'] for r in runs]):.3f} acc={np.mean([r['systems'][s]['accuracy'] for r in runs]):.3f} "
          f"violations={np.mean([r['systems'][s]['semantic_violation_rate'] for r in runs]):.3f}")
e1 = {"experiment": "E1", "measures": "agreement with synthetic generator convention on unseen schemas", "meta": E.run_meta({"mode": MODE, "split_seeds": SEEDS, "n_items": len(syn)}), "runs": runs}
json.dump(e1, open(f"{OUT}/reports/e1.json", "w"), indent=1)

## Ablation - leave one feature group out (hybrid, same split)

In [ ]:
asg = S.grouped_stratified_split(syn, group=lambda x: x["db_id"], seed=SEEDS[0])
tri = [idx[i["item_id"]] for i in syn if asg[i["item_id"]] == "train"]; tei = [idx[i["item_id"]] for i in syn if asg[i["item_id"]] == "test"]
ab = E.ablate(X[tri], [syn[k]["label"] for k in tri], X[tei], [syn[k]["label"] for k in tei], seed=0)
print("full", round(ab["full"], 3))
for g, v in ab["groups"].items(): print(f"{g:18s} n={v['n_features']:2d} macroF1_without={v['macro_f1_without']:.3f} delta={v['delta_vs_full']:+.3f}")
json.dump({"experiment": "ablation_leave_one_group_out", "meta": E.run_meta({"mode": MODE}), **ab}, open(f"{OUT}/reports/ablation.json", "w"), indent=1)

## E2 - synthetic-trained -> real Spider queries judged by LLM annotators (transfer; NOT human accuracy)

In [ ]:
labp = f"{OUT}/llm-labels/pilot_llm_labels.jsonl"
if os.path.exists(labp) and os.path.exists(f"{OUT}/cc-by-sa-4.0/features_real.npz"):
    fr = P.load_features(f"{OUT}/cc-by-sa-4.0/features_real.npz")
    real = {json.loads(l)["item_id"]: json.loads(l) for l in open(f"{OUT}/cc-by-sa-4.0/real_items.jsonl")}
    pos = {i: k for k, i in enumerate(fr["item_ids"])}
    labs = [json.loads(l) for l in open(labp)]
    ev = [l for l in labs if l["status"] == "agree" and not l["reserved_for_human_validation"] and l["item_id"] in pos]
    print(len(ev), "LLM-consensus items available (reserved human-validation items excluded)")
    if len(ev) >= 20:
        test = [dict(real[l["item_id"]], label=l["label"], acceptable=set(l["acceptable"])) for l in ev]
        k = [pos[l["item_id"]] for l in ev]
        r2 = E.evaluate_all(syn, test, seed=0, Xtr=X, Xte=fr["table"][k], cols_te=[fr["columns"][i] for i in k])
        print("test majority", r2["test_majority"])
        for s in E.SYSTEMS: print(f"{s:16s} acc={r2['systems'][s]['accuracy']:.3f} violations={r2['systems'][s]['semantic_violation_rate']:.3f}")
        json.dump({"experiment": "E2", "measures": "agreement with LLM consensus on real queries; not human", "meta": E.run_meta({"mode": MODE, "n": len(ev)}), **r2}, open(f"{OUT}/reports/e2.json", "w"), indent=1)
    else: print("pilot items not in this run's subset; skipping E2 (use MODE=full)")
else: print("no pilot labels / real features: E2 skipped")